# Milestone 2: Data Quality dan Cleaning

Notebook ini melanjutkan profiling CommonLingua pada Milestone 1. Data mentah berupa dua CSV (`train` dan `validation`) dengan kolom teks, kode bahasa, sumber, identifier, dan metadata. File mentah tidak diubah; hasil bersih disimpan terpisah sebagai Parquet di `data/processed/`.

Keputusan cleaning: `text` dan `lang` wajib tersedia; null pada metadata opsional dipertahankan agar tidak mengarang informasi. Panjang teks dianalisis sebagai outlier, tetapi teks tidak dipotong atau dihapus hanya karena panjang. Deduplikasi memakai kombinasi `lang`, `source`, dan `identifier` yang tidak null; baris tanpa identifier dipertahankan.

In [1]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
DATA_FILES = {
    'train': DATA_DIR / 'commonlingua_train.csv',
    'validation': DATA_DIR / 'commonlingua_validation.csv',
}
OUTPUT_FILES = {
    split: OUTPUT_DIR / f'commonlingua_{split}_clean.parquet'
    for split in DATA_FILES
}

missing_files = [path for path in DATA_FILES.values() if not path.exists()]
if missing_files:
    missing_list = '\n'.join(str(path) for path in missing_files)
    raise FileNotFoundError(
        f'File dataset tidak ditemukan:\n{missing_list}\n'
        'Pastikan CSV tersedia di data/raw/.'
    )

for split, path in DATA_FILES.items():
    print(f'{split}: {path.name} ({path.stat().st_size / 1_000_000_000:.2f} GB)')

train: commonlingua_train.csv (2.06 GB)
validation: commonlingua_validation.csv (0.23 GB)


## 1. Baca Data dan Profiling DuckDB

Polars membaca CSV secara lazy supaya file besar tidak langsung dimuat seluruhnya ke memori. `SUMMARIZE` DuckDB dijalankan langsung pada file untuk melihat statistik setiap kolom.

In [2]:
def scan_dataset(path: Path) -> pl.LazyFrame:
    return pl.scan_csv(
        path,
        infer_schema=False,
        null_values=['', 'NULL', 'null'],
        low_memory=True,
    )


lazy_data = {split: scan_dataset(path) for split, path in DATA_FILES.items()}
connection = duckdb.connect()
profil_duckdb = {}

for split, path in DATA_FILES.items():
    relation = connection.sql(
        'SUMMARIZE SELECT * FROM read_csv_auto(?, strict_mode=false, parallel=false)',
        params=[str(path)],
    )
    profil_duckdb[split] = pl.DataFrame(
        relation.fetchall(),
        schema=relation.columns,
        orient='row',
    )
    print(f'Profil DuckDB {split}:')
    display(profil_duckdb[split])

for split, frame in lazy_data.items():
    print(f'Skema Polars {split}: {frame.collect_schema()}')

Profil DuckDB train:


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,null,null,null,null,null,i64,"decimal[38,2]"
"""text""","""VARCHAR""","""…","""𱎼家话，又叫重安江苗语，是汉藏语系苗瑶语族一種闲话。 张济…",2323149,null,null,null,null,null,2482568,0.00
"""lang""","""VARCHAR""","""abk""","""zul""",350,null,null,null,null,null,2482568,0.00
"""source""","""VARCHAR""","""Anno""","""usc""",117,null,null,null,null,null,2482568,0.00
"""identifier""","""VARCHAR""","""-0xVHB9387w_1""","""（××××）××执异字第××号""",1901901,null,null,null,null,null,2482568,0.56
"""title""","""VARCHAR""",""" DARDUMAR VOA: Yadda Matashiya…","""󠀦󠀢󠀤󠀙󠀜󠀚󠀛-󠀦󠀢󠀤󠀙󠀜󠀚󠀛""",1574942,null,null,null,null,null,2482568,1.90
"""collection""","""VARCHAR""","""Achaemenid Royal Inscriptions …","""usc""",246,null,null,null,null,null,2482568,1.17
"""license""","""VARCHAR""","""CC-BY-SA""","""Various open science""",14,null,null,null,null,null,2482568,1.08
"""open_type""","""VARCHAR""","""Open_Culture""","""Web""",2,null,null,null,null,null,2482568,1.08
"""creator""","""VARCHAR""","""Barat et al.""","""Masoretic""",2,null,null,null,null,null,2482568,99.93


Profil DuckDB validation:


column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,null,null,null,null,null,i64,"decimal[38,2]"
"""text""","""VARCHAR""","""  Abstract: T…","""𧒽崗站（粵拼：leoi1 gong1 zaam6，英文：Le…",277830,null,null,null,null,null,272875,0.00
"""lang""","""VARCHAR""","""abk""","""zul""",337,null,null,null,null,null,272875,0.00
"""source""","""VARCHAR""","""Anno""","""reg_docs""",89,null,null,null,null,null,272875,0.00
"""identifier""","""VARCHAR""","""0000213163_34""","""（2021）黔0502民初12481号""",254827,null,null,null,null,null,272875,0.57
"""title""","""VARCHAR""",""" Gaza, sigidadenw tina se ka s…","""𧒽崗站 (地鐵)""",234288,null,null,null,null,null,272875,1.91
"""collection""","""VARCHAR""","""Achaemenid Royal Inscriptions …","""riao/The Royal Inscriptions of…",181,null,null,null,null,null,272875,0.10
"""license""","""VARCHAR""","""CC-BY-SA""","""Various open science""",11,null,null,null,null,null,272875,0.00
"""open_type""","""VARCHAR""","""Open_Culture""","""Web""",2,null,null,null,null,null,272875,0.00
"""creator""","""VARCHAR""","""Barat et al.""","""Masoretic""",2,null,null,null,null,null,272875,99.93


Skema Polars train: Schema({'text': String, 'lang': String, 'source': String, 'identifier': String, 'title': String, 'collection': String, 'license': String, 'open_type': String, 'creator': String, 'date': String})
Skema Polars validation: Schema({'text': String, 'lang': String, 'source': String, 'identifier': String, 'title': String, 'collection': String, 'license': String, 'open_type': String, 'creator': String, 'date': String})


## 2. Missing Values dan Validasi Kolom Wajib

Kolom `text` dan `lang` menjadi kolom wajib. Null atau string kosong pada keduanya akan dikeluarkan. Nilai null pada metadata lain dipertahankan karena ketiadaan metadata bukan alasan untuk mengisi nilai tebakan.

In [3]:
KOLOM_WAJIB = ['text', 'lang']
missing_columns = {
    split: [column for column in KOLOM_WAJIB if column not in frame.collect_schema().names()]
    for split, frame in lazy_data.items()
}
invalid_schema = {split: columns for split, columns in missing_columns.items() if columns}
if invalid_schema:
    raise ValueError(f'Kolom wajib tidak ditemukan: {invalid_schema}')

profil_null = {}
for split, frame in lazy_data.items():
    columns = frame.collect_schema().names()
    null_counts = frame.select(
        pl.len().alias('jumlah_baris'),
        *[pl.col(column).null_count().alias(f'{column}_null') for column in columns],
    ).collect()
    profil_null[split] = null_counts
    print(f'Jumlah null {split}:')
    display(null_counts)

data_valid = {}
jumlah_sebelum = {}
jumlah_setelah_missing = {}
for split, frame in lazy_data.items():
    jumlah_sebelum[split] = int(frame.select(pl.len()).collect().item())
    wajib_terisi = pl.all_horizontal(
        [
            pl.col(column).is_not_null()
            & (pl.col(column).str.strip_chars() != '')
            for column in KOLOM_WAJIB
        ]
    )
    data_valid[split] = frame.filter(wajib_terisi)
    jumlah_setelah_missing[split] = int(
        data_valid[split].select(pl.len()).collect().item()
    )
    print(
        f'{split}: dikeluarkan karena text/lang kosong = '
        f'{jumlah_sebelum[split] - jumlah_setelah_missing[split]:,}'
    )

Jumlah null train:


jumlah_baris,text_null,lang_null,source_null,identifier_null,title_null,collection_null,license_null,open_type_null,creator_null,date_null
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
2482568,0,0,0,13836,47122,28989,26693,26693,2480877,56236


Jumlah null validation:


jumlah_baris,text_null,lang_null,source_null,identifier_null,title_null,collection_null,license_null,open_type_null,creator_null,date_null
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
272875,0,0,0,1558,5214,270,0,0,272694,3228


train: dikeluarkan karena text/lang kosong = 0
validation: dikeluarkan karena text/lang kosong = 0
